# Retrieval-Augmented Generation (RAG) — v2

<sup>This notebook is a part of the Natural Language Processing class at the University of Ljubljana, Faculty of Computer and Information Science.</sup>

---

Large Language Models are remarkable at reasoning, but they have one fundamental limitation: **their knowledge is frozen at the time of training**. Once trained, the model cannot learn new facts unless it is retrained or fine-tuned — both expensive operations.

**Retrieval-Augmented Generation (RAG)** solves this by giving the model access to an external knowledge base *at inference time*. Instead of relying solely on memorised parameters, the model can look up relevant passages and ground its answer in real documents.

> **Analogy.** RAG is to an LLM what an **open-book exam** is to a student. The student (model) is not expected to memorise every fact; instead, they are allowed to consult textbooks (knowledge base) and reason from what they find.

---

## What you will learn

| Section | Topic |
|---|---|
| 1 | The hallucination problem: why pure LLMs fail on factual questions |
| 2 | Loading and inspecting a knowledge base (Wikipedia) |
| 3 | Text splitting strategies and why chunk size matters |
| 4 | Embeddings: turning text into vectors |
| 5 | Vector stores and FAISS indexing |
| 6 | Retrieval: similarity search vs. MMR |
| 7 | Building a RAG chain with modern **LCEL** syntax |
| 8 | Without RAG vs. with RAG — a direct comparison |
| 9 | Conversational RAG with memory |
| 10 | Exercises |

---

## RAG Architecture

The RAG pipeline has two phases:

**Indexing (done once, offline)**
```
Documents  →  Text Splitter  →  Embedding Model  →  Vector Store
```

**Retrieval + Generation (done at query time)**
```
User Query  →  Embedding Model  →  Vector Store (search)  →  Top-k Chunks
                                                                    ↓
                                          Prompt = [System + Context + Query]  →  LLM  →  Answer
```

The key insight is that **the query and documents are embedded into the same vector space**, so semantic similarity can be measured with cosine distance, regardless of exact word overlap.

## 1  Setup

We need the following libraries:

| Library | Purpose |
|---|---|
| `transformers` | Model loading and text generation pipeline |
| `bitsandbytes` | 4-bit quantisation (reduces GPU memory from ~16 GB to ~5 GB) |
| `accelerate` | Device placement and mixed-precision support |
| `langchain` / `langchain-community` | Document loaders, chains, retrievers |
| `langchain-huggingface` | LangChain ↔ HuggingFace bridge |
| `sentence-transformers` | Embedding models |
| `faiss-cpu` | Fast vector similarity search |
| `wikipedia` | Wikipedia document loader |
| `tiktoken` | Token counting (used by some text splitters) |

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate
!pip install -q -U langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface
!pip install -q -U sentence-transformers faiss-cpu wikipedia tiktoken

In [ ]:
# Authenticate to access gated models (Meta Llama-3)
# You need a HuggingFace account and must accept the licence at huggingface.co/meta-llama
!huggingface-cli login

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version : 2.10.0+cu129
CUDA available  : True
GPU             : NVIDIA A100-SXM4-40GB
VRAM            : 42.4 GB


## 2  The Problem: LLMs Hallucinate Facts

Before adding any retrieval, let's demonstrate the core problem that RAG solves.

LLMs generate text by predicting the next token based on patterns in training data. When asked a factual question about something they don't know well, they don't say "I don't know" — they **hallucinate** plausible-sounding but wrong answers.

We will use **Meta Llama-3-8B-Instruct** as our LLM. We load it in **4-bit quantisation** so it fits in ~5 GB of VRAM instead of the full 16 GB.

### Why 4-bit Quantisation?

Model weights are stored as floating-point numbers. By default, modern models use **bfloat16** (2 bytes per parameter). An 8B-parameter model therefore needs 16 GB just for weights.

**NF4 (NormalFloat 4-bit)** quantisation stores each weight in 4 bits (0.5 bytes), reducing memory by **4×** with very little quality loss, because weight distributions in neural networks are approximately normal — NF4 is specifically designed to minimise the error for this distribution.

In [2]:
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — optimal for bell-curve weight distributions
    bnb_4bit_use_double_quant=True,     # Quantise the quantisation constants too (saves ~0.4 bits/param)
    bnb_4bit_compute_dtype=torch.bfloat16,  # Use bfloat16 for matrix multiplications
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",          # Automatically distribute across available GPUs
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # Left padding for inference: generated tokens follow the last real token, not padding tokens
print("Model loaded.")

Loading model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded.


In [71]:
from transformers import GenerationConfig
from langchain_huggingface import HuggingFacePipeline

eot_token_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

hf_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=2048,   
    do_sample=False,
    repetition_penalty=1.1,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=[tokenizer.eos_token_id, eot_token_id]
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

In [72]:
# apply_chat_template reads the chat format directly from the tokenizer config,
# so this works for Llama-3, Mistral, Qwen, or any other model without changes.

def make_plain_prompt(question: str) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the question concisely and accurately."},
        {"role": "user",   "content": question},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

plain_chain = RunnableLambda(make_plain_prompt) | llm | StrOutputParser()

In [73]:
# Ask a highly specific factual question — something the model may not know well
question = "What are the key differences between BERT and RoBERTa in terms of training procedure?"

print("=== WITHOUT RAG ===")
answer_no_rag = plain_chain.invoke({"question": question})
print(answer_no_rag)

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== WITHOUT RAG ===
The key differences between BERT (Bidirectional Encoder Representations from Transformers) and RoBERTa (Robustly Optimized BERT Pretraining Approach) in terms of training procedure are:

1. **Next Sentence Prediction**: BERT was trained with a next sentence prediction task, where it predicts whether two sentences are adjacent or not. RoBERTa does not use this task.
2. **Masked Language Modeling**: Both models use masked language modeling, but RoBERTa uses a different approach to mask tokens. RoBERTa randomly selects 15% of the input tokens for masking, whereas BERT uses a fixed 15% rate.
3. **Optimizer and Learning Rate**: RoBERTa uses the AdamW optimizer and a learning rate schedule that is more aggressive than BERT's.
4. **Batch Size**: RoBERTa uses a larger batch size (256) compared to BERT (128).
5. **Number of Layers**: RoBERTa has the same number of layers as BERT (24), but its layer sizes are slightly smaller.

These changes allowed RoBERTa to outperform BERT

In [74]:
# Ask a highly specific factual question — something the model may not know well
question = "In what year was the founding stallion Siglavy foaled and where did he originally come from?"

print("=== WITHOUT RAG ===")
answer_no_rag = plain_chain.invoke({"question": question})
print(answer_no_rag)

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== WITHOUT RAG ===
The founding stallion Siglavy was foaled in 1835 and originated from Hungary.


The model may give a reasonable-sounding answer, but it is reconstructing facts from memory — which can contain errors or omissions. Now let's build a RAG system that **retrieves the actual source text** before answering.

## 3  Building the Knowledge Base

### 3.1  Loading Documents

We use **Wikipedia** as our knowledge source. It has two advantages for demos:

1. No authentication or scraping needed — `WikipediaLoader` fetches articles via the Wikipedia API.
2. The content is well-structured and factual, making it easy to verify whether the model's answer is grounded.

We load several NLP-related articles to give our RAG system a rich knowledge base.

In [75]:
# Articles to include in our knowledge base
topics = [
    "BERT (language model)",
    "GPT (language model)",
    "Transformer (deep learning architecture)",
    "Word2vec",
    "Attention mechanism",
    "Transfer learning",
    "Natural language processing",
    "Lipizzan",
    "Primož Trubar",
    "Idrija mercury mine",
]

print("Fetching Wikipedia articles...")
all_docs = []
for topic in topics:
    loader = WikipediaLoader(query=topic, load_max_docs=1, doc_content_chars_max=30000)
    docs = loader.load()
    all_docs.extend(docs)
    print(f"  Loaded: '{topic}' ({len(docs[0].page_content)} chars)")

print(f"\nTotal documents: {len(all_docs)}")
print(f"Total characters: {sum(len(d.page_content) for d in all_docs):,}")

Fetching Wikipedia articles...
  Loaded: 'BERT (language model)' (17538 chars)
  Loaded: 'GPT (language model)' (18833 chars)
  Loaded: 'Transformer (deep learning architecture)' (30000 chars)
  Loaded: 'Word2vec' (29636 chars)
  Loaded: 'Attention mechanism' (25514 chars)
  Loaded: 'Transfer learning' (8663 chars)
  Loaded: 'Natural language processing' (30000 chars)
  Loaded: 'Lipizzan' (22504 chars)
  Loaded: 'Primož Trubar' (7430 chars)
  Loaded: 'Idrija mercury mine' (6939 chars)

Total documents: 10
Total characters: 197,057


In [76]:
# Inspect one document
print("=== Document metadata ===")
print(all_docs[0].metadata)
print("\n=== First 500 characters ===")
print(all_docs[0].page_content[:500])

=== Document metadata ===
{'title': 'BERT (language model)', 'summary': 'Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved the state of the art for large language models. As of 2020, BERT is a ubiquitous baseline in natural language processing (NLP) experiments. \nBERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It found applications for many natural language processing tasks, such as coreference resolution and polysemy resolution. It improved on ELMo and spawned the study of "BERTology", which attempts to interpret what is learned by BERT.\nBERT was originally implemented in the English language a

### 3.2  Text Splitting

Embedding models and LLM context windows both have **token limits**. A full Wikipedia article may have 10,000+ characters — too long to embed as a single unit or include in a prompt wholesale.

We must split documents into **chunks**: shorter, self-contained passages that can be:
1. Embedded individually into vectors.
2. Passed selectively to the LLM based on relevance.

**Key parameters:**

| Parameter | Effect |
|---|---|
| `chunk_size` | Maximum size of each chunk (in characters or tokens) |
| `chunk_overlap` | How many characters from the end of one chunk are repeated at the start of the next |

**Why overlap?** Important context may straddle a chunk boundary. Overlap ensures that such information appears in at least one complete chunk. Too much overlap wastes space; too little risks missing cross-boundary context.

**Choosing chunk size:**
- **Too small** → chunks may lack enough context to answer a question; retrieval may be noisy.
- **Too large** → fewer, denser chunks; important passages are diluted by surrounding text; may overflow embedding model limits.
- A common starting point: **512–1024 characters** (roughly 100–200 tokens).

In [110]:
# RecursiveCharacterTextSplitter tries to split on natural boundaries first:
# paragraphs (\n\n) → sentences (\n) → words ( ) → characters
# This preserves semantic coherence better than a hard character cut.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,          # Target size in characters
    chunk_overlap=100,       # 100-char overlap between consecutive chunks
    length_function=len,
    add_start_index=True,    # Store the original char offset in metadata
)

chunks = text_splitter.split_documents(all_docs)

print(f"Documents → chunks: {len(all_docs)} → {len(chunks)}")
print(f"Average chunk size : {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
print(f"Min / Max          : {min(len(c.page_content) for c in chunks)} / {max(len(c.page_content) for c in chunks)} chars")

Documents → chunks: 10 → 389
Average chunk size : 518 chars
Min / Max          : 5 / 799 chars


### 3.3  Comparing Chunk Sizes

To build intuition, let's see how different `chunk_size` values affect the number and character of the chunks.

In [112]:
print(f"{'chunk_size':>12}  {'num_chunks':>10}  {'avg_len':>8}")
print("-" * 36)
for size in [200, 400, 800, 1600]:
    sp = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=size // 8)
    ch = sp.split_documents(all_docs)
    avg = sum(len(c.page_content) for c in ch) / len(ch)
    print(f"{size:>12}  {len(ch):>10}  {avg:>8.0f}")

  chunk_size  num_chunks   avg_len
------------------------------------
         200        1381       146
         400         753       272
         800         389       518
        1600         186      1080


## 4  Embeddings

An **embedding model** maps text to a fixed-size dense vector. Texts with similar meaning end up close together in this high-dimensional space, enabling semantic (meaning-based) search rather than keyword matching.

```
"BERT uses masked language modelling"  →  [0.12, -0.87, 0.33, ...]
"BERT is trained with MLM"             →  [0.14, -0.85, 0.31, ...]   ← close!
"The weather is sunny today"           →  [-0.9,  0.23, 0.71, ...]   ← far away
```

We use `sentence-transformers/all-MiniLM-L6-v2` — a lightweight model (22M parameters) that produces 384-dimensional embeddings and runs fast even on CPU.

In [80]:
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # L2-normalise → cosine similarity = dot product
)

print(f"Embedding model loaded: {EMBED_MODEL}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [81]:
# Demonstrate semantic similarity directly on embeddings
import numpy as np

sentences = [
    "BERT uses a masked language model pre-training objective.",
    "BERT is trained by masking tokens and predicting them.",
    "The Eiffel Tower is located in Paris, France.",
]

vecs = embeddings.embed_documents(sentences)
vecs = np.array(vecs)  # shape: (3, 384)

# Cosine similarity matrix (vectors are already L2-normalised, so dot product = cosine)
sim = vecs @ vecs.T

print("Cosine similarity matrix:")
print(f"{'':50s}  S1    S2    S3")
labels = ["S1: BERT MLM objective", "S2: BERT masking tokens", "S3: Eiffel Tower"]
for i, label in enumerate(labels):
    row = "  ".join(f"{sim[i,j]:.3f}" for j in range(3))
    print(f"{label:50s}  {row}")

print("\nS1 and S2 are semantically similar → high cosine similarity.")
print("S3 is about a different topic → low similarity with S1/S2.")

Cosine similarity matrix:
                                                    S1    S2    S3
S1: BERT MLM objective                              1.000  0.804  -0.077
S2: BERT masking tokens                             0.804  1.000  -0.090
S3: Eiffel Tower                                    -0.077  -0.090  1.000

S1 and S2 are semantically similar → high cosine similarity.
S3 is about a different topic → low similarity with S1/S2.


## 5  Vector Store (FAISS)

A **vector store** indexes all embedded chunks so we can find the nearest neighbours of a query vector efficiently.

**FAISS** (Facebook AI Similarity Search) is a library for fast approximate nearest-neighbour search in high-dimensional spaces. For our use case:

1. Each chunk is embedded into a 384-dimensional vector.
2. All vectors are stored in FAISS.
3. At query time: embed the query → search FAISS → return the top-k most similar chunk vectors.

This is **much faster** than computing cosine similarity against every stored vector sequentially.

[FAISS and langchain docs](https://docs.langchain.com/oss/python/integrations/vectorstores/faiss)

In [115]:
print(f"Indexing {len(chunks)} chunks into FAISS...")
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Done.")
print(f"Index size: {vectorstore.index.ntotal} vectors of dimension {vectorstore.index.d}")

Indexing 389 chunks into FAISS...
Done.
Index size: 389 vectors of dimension 384


## 6  Retrieval: Similarity Search vs. MMR

The retriever is the component that takes a query and returns the most relevant chunks.

### 6.1  Similarity Search

Plain **similarity search** returns the top-k chunks with the highest cosine similarity to the query. Simple and effective, but it can return **redundant** results: if the top-5 chunks all discuss the same narrow sub-topic, important tangential context is missed.

### 6.2  Maximum Marginal Relevance (MMR)

**MMR** balances relevance and diversity. It selects chunks that are:
- **Relevant** to the query (high similarity to query)
- **Diverse** from already-selected chunks (low similarity to each other)

The trade-off is controlled by `lambda_mult` (0 = max diversity, 1 = max relevance).

MMR is better when you want broad coverage of a topic rather than several chunks that all say the same thing.

In [116]:
query = "How does BERT differ from GPT in terms of training objectives?"

# --- Similarity search ---
sim_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)
sim_results = sim_retriever.invoke(query)

print("=== Similarity Search Results ===")
for i, doc in enumerate(sim_results):
    title = doc.metadata.get("title", "unknown")
    print(f"[{i+1}] Source: {title}")
    print(doc.page_content[:200])
    print()

=== Similarity Search Results ===
[1] Source: BERT (language model)
BERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It fo

[2] Source: BERT (language model)
== Interpretation ==
Language models like ELMo, GPT-2, and BERT, spawned the study of "BERTology", which attempts to interpret what is learned by these models. Their performance on these natural langu

[3] Source: Generative pre-trained transformer
This phenomenon has influenced the development of larger GPT models and contributed to their increased effectiveness across a wide range of tasks.

[4] Source: BERT (language model)
The high performance of the BERT model could also be attributed to the fact that it is bidirectionally trained. This means that BERT, based on the Transformer model architecture, applies its self-atte



In [121]:
# --- MMR search ---
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5},
    # fetch_k: initial candidate pool size (larger = better diversity quality)
    # lambda_mult: relevance weight (0.5 = equal balance)
)
mmr_results = mmr_retriever.invoke(query)

print("=== MMR Search Results ===")
for i, doc in enumerate(mmr_results):
    title = doc.metadata.get("title", "unknown")
    print(f"[{i+1}] Source: {title}")
    print(doc.page_content[:200])
    print()

=== MMR Search Results ===
[1] Source: BERT (language model)
BERT is trained by masked token prediction and next sentence prediction. With this training, BERT learns contextual, latent representations of tokens in their context, similar to ELMo and GPT-2. It fo

[2] Source: Generative pre-trained transformer
This phenomenon has influenced the development of larger GPT models and contributed to their increased effectiveness across a wide range of tasks.

[3] Source: BERT (language model)
=== Cost ===
BERT was trained on the BookCorpus (800M words) and a filtered version of English Wikipedia (2,500M words) without lists, tables, and headers.
Training BERTBASE  on 4 cloud TPU (16 TPU ch

[4] Source: Generative pre-trained transformer
EinsteinGPT – for sales and marketing domains, to aid with customer relationship management (uses GPT-3.5)
BloombergGPT – for the financial domain, to aid with financial news and information (uses "fr



In [122]:
# Count how many distinct source articles each retrieval strategy returns
sim_sources = {d.metadata.get("title", "?") for d in sim_results}
mmr_sources = {d.metadata.get("title", "?") for d in mmr_results}

print(f"Similarity search: {len(sim_sources)} distinct sources → {sim_sources}")
print(f"MMR search       : {len(mmr_sources)} distinct sources → {mmr_sources}")
print("\nMMR typically returns results from more diverse sources.")

Similarity search: 2 distinct sources → {'Generative pre-trained transformer', 'BERT (language model)'}
MMR search       : 2 distinct sources → {'Generative pre-trained transformer', 'BERT (language model)'}

MMR typically returns results from more diverse sources.


## 7  Building the RAG Chain with LCEL

**LangChain Expression Language (LCEL)** is LangChain's modern way of composing chains using the pipe operator `|`. Each component receives output from the previous step:

```python
chain = step_1 | step_2 | step_3
chain.invoke(input)  # passes input through all steps
```

This replaces the older `ConversationalRetrievalChain` and similar high-level abstractions, which are now deprecated. LCEL is more transparent, composable, and supports streaming out of the box.

### Full RAG chain

```
Question
   ├──────────────────────────────────► RunnablePassthrough (keeps question)
   │                                             │
   └──► Retriever ──► format_docs (join)         │
                           │                     │
                    {context: ..., question: ...} │
                                    ↓
                               PromptTemplate
                                    ↓
                                   LLM
                                    ↓
                              StrOutputParser
```

In [123]:
# Choose our retriever (MMR for diversity)
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.6},
)

# Helper to concatenate retrieved Documents into a single context string
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Source: {d.metadata.get('title', 'unknown')}]\n{d.page_content}"
        for d in docs
    )

In [87]:
# RAG prompt — built via apply_chat_template so the format is always correct for the loaded model

def make_rag_prompt(inputs: dict) -> str:
    system = (
        "You are a precise, helpful assistant. Answer the question using ONLY the context below.\n"
        "If the answer is not in the context, say \"I don't have enough information to answer that.\"\n"
        "Do not make up facts. Cite which source(s) you used at the end of your answer.\n\n"
        f"CONTEXT:\n{inputs['context']}"
    )
    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": inputs["question"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

RAG_PROMPT = RunnableLambda(make_rag_prompt)

In [88]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

In [89]:
# Helper to print the answer neatly
def ask(question: str):
    print(f"Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(answer)
    print("=" * 60)
    return answer

In [90]:
ask("What are the key differences between BERT and GPT in terms of their architecture and training objectives?")

Question: What are the key differences between BERT and GPT in terms of their architecture and training objectives?
------------------------------------------------------------


Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Based on the provided context, I can answer the question within the given sources.

According to the sources, the key differences between BERT and GPT are:

* Architecture: BERT is an "encoder-only" transformer architecture, whereas GPT is not explicitly mentioned as having a specific architecture description in the provided context.
* Training Objectives: BERT is trained on two main objectives: masked token prediction and next sentence prediction. On the other hand, GPT is not explicitly mentioned as having a specific training objective in the provided context.

Note that the provided context does not provide detailed information about GPT's architecture or training objectives beyond mentioning its influence on the development of larger GPT models. Therefore, I do not have enough information to provide more specific details about GPT's architecture and training objectives.

Sources:
1. [Source: BERT (language model)]
2. [Source: Generative pre-trained transformer]


'Based on the provided context, I can answer the question within the given sources.\n\nAccording to the sources, the key differences between BERT and GPT are:\n\n* Architecture: BERT is an "encoder-only" transformer architecture, whereas GPT is not explicitly mentioned as having a specific architecture description in the provided context.\n* Training Objectives: BERT is trained on two main objectives: masked token prediction and next sentence prediction. On the other hand, GPT is not explicitly mentioned as having a specific training objective in the provided context.\n\nNote that the provided context does not provide detailed information about GPT\'s architecture or training objectives beyond mentioning its influence on the development of larger GPT models. Therefore, I do not have enough information to provide more specific details about GPT\'s architecture and training objectives.\n\nSources:\n1. [Source: BERT (language model)]\n2. [Source: Generative pre-trained transformer]'

In [91]:
ask("What is the attention mechanism and why was it introduced?")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the attention mechanism and why was it introduced?
------------------------------------------------------------
According to the provided sources, the attention mechanism is a method that determines the importance of each component in a sequence relative to the other components in that sequence. In natural language processing, this importance is represented by "soft" weights assigned to each word in a sentence.

The attention mechanism was introduced to address the limitations of traditional recurrent neural networks (RNNs), which struggle to capture long-range dependencies in sequential data. By allowing the model to focus on specific parts of the input sequence, attention enables the network to selectively weigh the importance of different elements and better capture complex relationships between them.

Sources:
* [Source: Attention (machine learning)]


'According to the provided sources, the attention mechanism is a method that determines the importance of each component in a sequence relative to the other components in that sequence. In natural language processing, this importance is represented by "soft" weights assigned to each word in a sentence.\n\nThe attention mechanism was introduced to address the limitations of traditional recurrent neural networks (RNNs), which struggle to capture long-range dependencies in sequential data. By allowing the model to focus on specific parts of the input sequence, attention enables the network to selectively weigh the importance of different elements and better capture complex relationships between them.\n\nSources:\n* [Source: Attention (machine learning)]'

In [92]:
ask("How does Word2vec represent words as vectors?")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: How does Word2vec represent words as vectors?
------------------------------------------------------------
According to the provided context, Word2vec represents a word as a high-dimensional vector of numbers which capture relationships between words. Specifically, words which appear in similar contexts are mapped to vectors which are nearby as measured by cosine similarity.


'According to the provided context, Word2vec represents a word as a high-dimensional vector of numbers which capture relationships between words. Specifically, words which appear in similar contexts are mapped to vectors which are nearby as measured by cosine similarity.'

In [93]:
ask("In what year was the founding stallion Siglavy foaled and where did he originally come from?")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: In what year was the founding stallion Siglavy foaled and where did he originally come from?
------------------------------------------------------------
According to the provided context, Siglavy, a gray Arabian stallion, was originally from Syria and was foaled in 1810.


'According to the provided context, Siglavy, a gray Arabian stallion, was originally from Syria and was foaled in 1810.'

## 8  Without RAG vs. With RAG — Side-by-Side

This is the core experiment. We ask the same specific factual question to:
1. The plain LLM (no context)
2. The RAG-augmented LLM (with retrieved Wikipedia context)

After running both cells, compare:
- **Specificity**: Does the RAG answer include concrete details absent from the baseline?
- **Grounding**: Does the RAG answer cite sources and stick to them?
- **Accuracy**: Is either answer factually wrong?

In [94]:
comparison_question = "In what year was the founding stallion Siglavy foaled and where did he originally come from?"

print("=" * 60)
print("WITHOUT RAG (pure LLM memory)")
print("=" * 60)
baseline = plain_chain.invoke({"question": comparison_question})
print(baseline)

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITHOUT RAG (pure LLM memory)
The founding stallion Siglavy was foaled in 1835 and originated from Hungary.


In [95]:
print("=" * 60)
print("WITH RAG (retrieved Wikipedia context)")
print("=" * 60)
rag_answer = rag_chain.invoke(comparison_question)
print(rag_answer)

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITH RAG (retrieved Wikipedia context)
According to the provided context, Siglavy, a gray Arabian stallion, was originally from Syria and was foaled in 1810.


In [124]:
# Also show what context was actually retrieved
retrieved = retriever.invoke(comparison_question)
print("=" * 60)
print(f"Retrieved {len(retrieved)} chunks:")
print("=" * 60)
for i, doc in enumerate(retrieved):
    print(f"\n[Chunk {i+1}] Source: {doc.metadata.get('title', 'unknown')}")
    print(doc.page_content[:1000], "...")

Retrieved 5 chunks:

[Chunk 1] Source: Lipizzan
Pluto: a gray Spanish stallion from the Royal Danish Stud, foaled in 1765
Conversano: a black Neapolitan stallion, foaled in 1767
Maestoso: a gray stallion from the Kladrub stud with a Spanish dam, foaled 1773, descendants today all trace via Maestoso X, foaled in Hungary in 1819
Favory: a dun stallion from the Kladrub stud, foaled in 1779
Neapolitano: a bay Neapolitan stallion from the Polesine, foaled in 1790
Siglavy: a gray Arabian stallion, originally from Syria, foaled in 1810
Two additional stallion lines are found in Croatia, Hungary, and other eastern European countries, as well as in North America. They are accepted as equal to the six classical lines by the Lipizzan International Federation. These are: ...

[Chunk 2] Source: Lipizzan
The methods for training the Lipizzan stallions at the Spanish Riding School were passed down via an oral tradition until Field Marshal Franz Holbein and Johann Meixner, Senior Rider at the School, 

## 9  Conversational RAG

So far our RAG chain handles single-turn questions. But real assistants need to maintain conversation history — a follow-up question like "Tell me more about that" only makes sense if the model remembers what "that" referred to.

**The challenge:** If the user says "How was it trained?" in a follow-up, the retriever receives this ambiguous query and may retrieve irrelevant chunks. We need to **reformulate** the question using the chat history before retrieval.

The modern LangChain approach uses two chains:
1. **History-aware retriever**: rewrites the follow-up question into a standalone question using chat history.
2. **Question-answering chain**: answers the standalone question using retrieved context.

```
Chat History + Follow-up Question
          ↓
  Reformulation Prompt + LLM
          ↓
   Standalone Question
          ↓
      Retriever
          ↓
  RAG Answer Chain
```

In [97]:
# Step 1: Rewrite a follow-up question into a standalone question using chat history.
# Uses apply_chat_template so the format matches the loaded model.

def make_contextualize_prompt(inputs: dict) -> str:
    history_text = "\n".join(
        f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}"
        for m in inputs.get("chat_history", [])
    )
    user_content = (
        "Given the conversation below, reformulate the final question as a standalone question "
        "that can be understood without the history. Return ONLY the question, nothing else.\n\n"
        f"HISTORY:\n{history_text}\n\nQUESTION: {inputs['input']}"
    )
    messages = [
        {"role": "system", "content": "You reformulate follow-up questions into standalone questions."},
        {"role": "user",   "content": user_content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

contextualize_chain = RunnableLambda(make_contextualize_prompt) | llm | StrOutputParser()

def contextualized_question(input_dict: dict) -> str:
    """If there is chat history, rewrite the question; otherwise pass it through."""
    if input_dict.get("chat_history"):
        return contextualize_chain.invoke(input_dict)
    return input_dict["input"]

In [98]:
# QA prompt for conversational RAG — also uses apply_chat_template

def make_conv_rag_prompt(inputs: dict) -> str:
    system = (
        "You are a helpful NLP tutor. Answer the question using only the context below.\n"
        "If the answer is not in the context, say you don't know. Be concise.\n\n"
        f"CONTEXT:\n{inputs['context']}"
    )
    # Interleave the stored chat history with the new user message
    messages = [{"role": "system", "content": system}]
    for msg in inputs.get("chat_history", []):
        role = "user" if isinstance(msg, HumanMessage) else "assistant"
        messages.append({"role": role, "content": msg.content})
    messages.append({"role": "user", "content": inputs["input"]})
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

conversational_rag_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(contextualized_question) | retriever | format_docs
    )
    | RunnableLambda(make_conv_rag_prompt)
    | llm
    | StrOutputParser()
)

In [99]:
# Stateful conversation loop
chat_history = []

def chat_with_rag(user_message: str) -> str:
    # The chain now returns a string directly (pure LCEL, no wrapper dict)
    answer = conversational_rag_chain.invoke({
        "input": user_message,
        "chat_history": chat_history,
    })
    # Append the exchange to history for next turn
    chat_history.append(HumanMessage(content=user_message))
    chat_history.append(AIMessage(content=answer))
    return answer

In [100]:
# Turn 1: Initial question
q1 = "What is the Transformer architecture?"
print(f"User: {q1}")
a1 = chat_with_rag(q1)
print(f"Assistant: {a1}")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: What is the Transformer architecture?
Assistant: According to the provided context, the Transformer architecture is described as having the following primary components:

* Encoder and Decoder
* Encoding layers that process all input tokens together
* Decoding layers that iteratively process the encoder's output and the decoder's output tokens so far
* Feedforward Network (FFN) modules, which are 2-layered multilayer perceptrons.


In [101]:
# Turn 2: Follow-up that references the previous answer
# Note: "it" refers to the Transformer — the system must use history to resolve this
q2 = "What problem was it designed to solve?"
print(f"User: {q2}")
a2 = chat_with_rag(q2)
print(f"Assistant: {a2}")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: What problem was it designed to solve?


Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: According to the context, the Transformer architecture was originally designed to improve machine translation, specifically to overcome limitations of previous architectures.


In [102]:
# Turn 3: Another follow-up
q3 = "How does the self-attention mechanism achieve this?"
print(f"User: {q3}")
a3 = chat_with_rag(q3)
print(f"Assistant: {a3}")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


User: How does the self-attention mechanism achieve this?


Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: I don't know. The context doesn't provide specific details on how the self-attention mechanism achieves this.


In [103]:
# Print the full conversation history
print("=== Full Conversation History ===")
for msg in chat_history:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    print(f"\n[{role}]: {msg.content[:300]}")

=== Full Conversation History ===

[User]: What is the Transformer architecture?

[Assistant]: According to the provided context, the Transformer architecture is described as having the following primary components:

* Encoder and Decoder
* Encoding layers that process all input tokens together
* Decoding layers that iteratively process the encoder's output and the decoder's output tokens so 

[User]: What problem was it designed to solve?

[Assistant]: According to the context, the Transformer architecture was originally designed to improve machine translation, specifically to overcome limitations of previous architectures.

[User]: How does the self-attention mechanism achieve this?

[Assistant]: I don't know. The context doesn't provide specific details on how the self-attention mechanism achieves this.


## 10  Inspecting What Gets Retrieved

Understanding *why* a RAG system gives a particular answer requires looking at the retrieved chunks. A retrieval-augmented answer is only as good as the chunks it receives — garbage in, garbage out.

Good diagnostics:
- Are the retrieved chunks from the correct source?
- Are they relevant to the question?
- Is there redundancy (same passage retrieved twice)?

In [104]:
def rag_with_sources(question: str):
    """Run RAG and print both the answer and the source chunks used."""
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    
    answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({
        "context": context,
        "question": question,
    })
    
    print(f"Question: {question}")
    print("-" * 60)
    print("ANSWER:")
    print(answer)
    print()
    print("RETRIEVED CONTEXT:")
    for i, doc in enumerate(retrieved):
        title = doc.metadata.get("title", "unknown")
        print(f"  [{i+1}] {title}: {doc.page_content[:150]}...")
    print("=" * 60)

rag_with_sources("What is transfer learning and how is it used in NLP?")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is transfer learning and how is it used in NLP?
------------------------------------------------------------
ANSWER:
According to the provided sources, transfer learning is a technique in machine learning where knowledge learned from one task is reused to improve performance on a different but related task. In the context of Natural Language Processing (NLP), transfer learning can be used to leverage knowledge gained from one NLP task to improve performance on another NLP task. For instance, knowledge gained from recognizing cars in images could be applied when trying to recognize trucks.

In NLP, transfer learning is often used to pre-train a model on a large, general-purpose dataset, and then fine-tune it on a smaller, task-specific dataset. This approach can help reduce the need for labeled data and speed up the development of new NLP models.

Sources:
- [Source: Transfer learning]
- [Source: Natural language processing]

Note: I did not find any specific information 

In [105]:
# Test with a question outside our knowledge base — the model should admit ignorance
rag_with_sources("What is the population of Buenos Aires?")

Both `max_new_tokens` (=2048) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the population of Buenos Aires?
------------------------------------------------------------
ANSWER:
I don't have enough information to answer that. The provided context does not mention Buenos Aires or its population.

RETRIEVED CONTEXT:
  [1] Attention (machine learning): where...
  [2] Generative pre-trained transformer: Gemini (initially based on their LaMDA family of conversation-trained language models, with plans to switch to PaLM)....
  [3] Lipizzan: The first evacuation of the twentieth century occurred in 1915 when the horses were evacuated from Lipica due to World War I and placed at Laxenburg a...
  [4] Idrija: == Church ==
The parish church in the town is dedicated to Saint Joseph the Worker and belongs to the Diocese of Koper. There are three other churches...
  [5] Word2vec: === Skip-gram ===...


## 11  Chunking Strategy Comparison

The choice of text splitter affects retrieval quality. Let's compare:

| Splitter | How it splits | Best for |
|---|---|---|
| `RecursiveCharacterTextSplitter` | Tries paragraph → sentence → char boundaries | General prose |
| `TokenTextSplitter` | Splits by token count (using a tokenizer) | Embedding models with token limits |

The `TokenTextSplitter` is more precise for models that have strict token-count limits, since character length and token count don't map 1:1 (a character with an accent may tokenise to 2 tokens).

In [106]:
# Token-based splitting
token_splitter = TokenTextSplitter(
    chunk_size=200,     # 200 tokens per chunk (~150 words)
    chunk_overlap=20,
)
token_chunks = token_splitter.split_documents(all_docs)

print(f"RecursiveCharacter (800 chars) : {len(chunks)} chunks")
print(f"Token (200 tokens)             : {len(token_chunks)} chunks")

# Show first chunk from each
print("\n--- RecursiveCharacter chunk ---")
print(chunks[0].page_content[:300])
print("\n--- Token chunk ---")
print(token_chunks[0].page_content[:300])

RecursiveCharacter (800 chars) : 389 chunks
Token (200 tokens)             : 389 chunks

--- RecursiveCharacter chunk ---
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved t

--- Token chunk ---
Bidirectional encoder representations from transformers (BERT) is a language model introduced in October 2018 by researchers at Google. It learns to represent text as a sequence of vectors using self-supervised learning. It uses the encoder-only transformer architecture. BERT dramatically improved t


In [107]:
# Build a second vectorstore with token-based chunks and compare retrieval
vectorstore_token = FAISS.from_documents(token_chunks, embeddings)
token_retriever = vectorstore_token.as_retriever(search_kwargs={"k": 4})

test_query = "How does BERT pre-training work?"

char_results = retriever.invoke(test_query)
tok_results  = token_retriever.invoke(test_query)

print("=== Character-based chunks ===")
for d in char_results:
    print(f"  {d.metadata.get('title','?')} | {len(d.page_content)} chars")

print("\n=== Token-based chunks ===")
for d in tok_results:
    print(f"  {d.metadata.get('title','?')} | {len(d.page_content)} chars")

=== Character-based chunks ===
  BERT (language model) | 572 chars
  BERT (language model) | 325 chars
  Transformer (deep learning) | 435 chars
  BERT (language model) | 430 chars
  BERT (language model) | 796 chars

=== Token-based chunks ===
  BERT (language model) | 962 chars
  BERT (language model) | 796 chars
  BERT (language model) | 1023 chars
  BERT (language model) | 947 chars


## 12  Summary

You have built a complete RAG system from scratch. Here is what each component does:

```
┌─────────────────────────────────────────────────────────────────┐
│                         INDEXING PHASE                          │
│                                                                 │
│  Documents  →  Text Splitter  →  Embeddings  →  Vector Store   │
│  (Wikipedia)   (chunk_size=800) (MiniLM-L6)    (FAISS)         │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│                        QUERY PHASE (LCEL)                       │
│                                                                 │
│  Question ──► Retriever (MMR, k=5)                              │
│       │              │                                          │
│       │        Top-k Chunks                                     │
│       │              │                                          │
│       └──────► Prompt Template ──► LLM ──► StrOutputParser      │
│                {context, question}   (Llama-3-8B-4bit)          │
└─────────────────────────────────────────────────────────────────┘
```

**Key design decisions and their effects:**

| Decision | Options | Impact |
|---|---|---|
| Chunk size | 200–2000 chars | Smaller = more precise; larger = more context per chunk |
| Overlap | 0–20% of chunk size | Reduces boundary artefacts |
| Embedding model | MiniLM (fast) → MPNet (accurate) → large models | Quality vs. speed |
| Retrieval type | similarity vs. MMR | Precision vs. diversity |
| k (num chunks) | 3–10 | More context = better answers, but higher latency and risk of dilution |
| LLM | Small (1B) → large (70B) | Answer quality vs. compute cost |